# Accessors — Advanced Problems with Solutions (Tutorial Style)

This notebook continues the same topic as the source notebook: using `__getattr__` and `__setattr__` when object fields are discovered dynamically from data rather than hard-coded into a class.

The teaching style is intentionally incremental. We will repeatedly use this pattern:

1. state a concrete problem,
2. make one small change,
3. run a short example,
4. explain what Python is doing,
5. then build the next feature on top of that observation.

We will use dictionaries as a simulated database so the focus stays on Python's attribute-access protocol.

## What we will build

We will start with a minimal dynamic record and progressively add:

- dynamic reads,
- dynamic writes,
- safe internal attributes,
- field inspection,
- read-only fields,
- deletion,
- audit history,
- snapshots,
- aliases,
- item access for unusual keys,
- explicit save/rollback,
- lazy relationships,
- relationship caching,
- computed fields,
- simple querying,
- serialization,
- and a compact final data layer.

The goal is not to replace a real ORM. The goal is to understand the mechanics well enough that the "magic" remains predictable.

## Simulated database

The outer dictionary contains tables. Each table maps a numeric record ID to a dictionary of field values.

In [1]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any

DB = {
    'Person': {
        1: {'first_name': 'Isaac', 'last_name': 'Newton', 'born': 1642, 'country_id': 1, 'active': True},
        2: {'first_name': 'Gottfried', 'last_name': 'von Leibniz', 'born': 1646, 'country_id': 5, 'active': True},
        3: {'first_name': 'Joseph', 'last_name': 'Fourier', 'born': 1768, 'country_id': 3, 'active': True},
        4: {'first_name': 'Bernhard', 'last_name': 'Riemann', 'born': 1826, 'country_id': 5, 'active': False},
    },
    'Country': {
        1: {'name': 'United Kingdom', 'capital': 'London', 'continent': 'Europe'},
        3: {'name': 'France', 'capital': 'Paris', 'continent': 'Europe'},
        5: {'name': 'Germany', 'capital': 'Berlin', 'continent': 'Europe'},
    },
}

ORIGINAL_DB = deepcopy(DB)
print(DB['Person'][1])

{'first_name': 'Isaac', 'last_name': 'Newton', 'born': 1642, 'country_id': 1, 'active': True}


# 1. Refresher — when does `__getattr__` run?

`__getattr__` is a fallback hook.

Python first tries normal attribute lookup. Only when that fails does it call `__getattr__`.

This matters because it lets methods, properties, and normal attributes keep their ordinary Python behavior.

In [2]:
class FallbackDemo:
    class_value = 'found normally'

    def __init__(self):
        self.instance_value = 'also found normally'

    def __getattr__(self, name):
        print(f'fallback for {name!r}')
        return f'<dynamic:{name}>'


demo = FallbackDemo()
print(demo.class_value)
print(demo.instance_value)
print(demo.unknown_name)

found normally
also found normally
fallback for 'unknown_name'
<dynamic:unknown_name>


Only the missing name reached `__getattr__`.

That is exactly why `__getattr__` is often a better fit than `__getattribute__` for dynamic record fields.

# Problem 1 — Read dictionary keys as attributes

Suppose a record arrives as a plain dictionary.

In [3]:
raw = {
    'first_name': 'Ada',
    'last_name': 'Lovelace',
    'born': 1815,
}

print(raw['first_name'])

Ada


We want this convenience instead:

```python
record.first_name
record.last_name
record.born
```

The fields are not declared in the class. They come from the dictionary at runtime.

## Step 1 — store the dictionary

For now we are not overriding `__setattr__`, so ordinary assignment during initialization is safe.

In [4]:
class DynamicRecordV1:
    def __init__(self, record):
        self._record = record

## Step 2 — use `__getattr__` as a fallback

If normal lookup cannot find the requested name, we look for that name in `_record`.

In [5]:
class DynamicRecordV1:
    def __init__(self, record):
        self._record = record

    def __getattr__(self, name):
        if name in self._record:
            return self._record[name]
        raise AttributeError(f'Unknown field: {name}')

In [6]:
ada = DynamicRecordV1(raw)

print(ada.first_name)
print(ada.last_name)
print(ada.born)

assert ada.first_name == 'Ada'
assert ada.born == 1815

Ada
Lovelace
1815


## Why raise `AttributeError`?

Because the operation is attribute lookup.

Functions such as `hasattr()` specifically expect missing attributes to fail with `AttributeError`.

In [7]:
print(hasattr(ada, 'first_name'))
print(hasattr(ada, 'middle_name'))

assert hasattr(ada, 'first_name')
assert not hasattr(ada, 'middle_name')

True
False


# Problem 2 — Add writes without accidental recursion

Now we want:

```python
record.last_name = 'King'
```

The dangerous part is that overriding `__setattr__` changes **every assignment** on the object, including assignments made in `__init__`.

Consider this broken idea:

```python
class Broken:
    def __init__(self, record):
        self._record = record

    def __setattr__(self, name, value):
        if name in self._record:
            self._record[name] = value
```

The first assignment in `__init__` already calls the custom setter, but `_record` does not exist yet.

## Step 1 — bypass our custom setter for internal state

The base implementation is available as `object.__setattr__`.

Inside custom access hooks, `object.__getattribute__` is also useful for direct access to trusted internal attributes.

In [8]:
class DynamicRecordV2:
    def __init__(self, record):
        object.__setattr__(self, '_record', record)

    def __getattr__(self, name):
        record = object.__getattribute__(self, '_record')
        if name in record:
            return record[name]
        raise AttributeError(name)

    def __setattr__(self, name, value):
        record = object.__getattribute__(self, '_record')
        if name in record:
            record[name] = value
            return
        raise AttributeError(f'Unknown field: {name}')

In [9]:
record = DynamicRecordV2({'name': 'old', 'count': 1})
record.name = 'new'
record.count = 2

print(record.name, record.count)
assert record.name == 'new'
assert record.count == 2

new 2


The important pattern is now visible:

- use `object.__setattr__` when bootstrapping internal state;
- use `object.__getattribute__` inside hooks when you need direct access to that state.

# Problem 3 — Separate internal names from public fields

A useful wrapper usually needs metadata such as `_record_id`, `_table_name`, or `_audit_log`.

Those are Python implementation details, not database fields.

We will use a simple convention:

> names beginning with `_` are internal attributes.

In [10]:
class DynamicRecordV3:
    def __init__(self, record, *, record_id=None, table_name=None):
        object.__setattr__(self, '_record', record)
        object.__setattr__(self, '_record_id', record_id)
        object.__setattr__(self, '_table_name', table_name)

    def __getattr__(self, name):
        record = object.__getattribute__(self, '_record')
        if name in record:
            return record[name]
        raise AttributeError(name)

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)
            return

        record = object.__getattribute__(self, '_record')
        if name in record:
            record[name] = value
            return

        raise AttributeError(f'Unknown public field: {name}')

In [11]:
person = DynamicRecordV3(
    {'first_name': 'Isaac', 'last_name': 'Newton'},
    record_id=1,
    table_name='Person',
)

person._debug_note = 'internal metadata'
person.last_name = 'Newton-Smith'

print(person._record_id)
print(person._table_name)
print(person._debug_note)
print(person.last_name)

1
Person
internal metadata
Newton-Smith


# Problem 4 — Discover the available fields

Dynamic attributes are convenient, but they are less obvious to inspect.

Let's expose the current field names through a property.

In [12]:
class InspectableRecord(DynamicRecordV3):
    @property
    def fields(self):
        record = object.__getattribute__(self, '_record')
        return tuple(record.keys())

In [13]:
person = InspectableRecord(DB['Person'][1])
print(person.fields)
assert 'first_name' in person.fields
assert 'country_id' in person.fields

('first_name', 'last_name', 'born', 'country_id', 'active')


## A name collision

What if the database itself contains a field named `fields`?

Normal Python lookup finds the property first, so the database value would be shadowed.

A robust dynamic API therefore benefits from an explicit field-access method.

In [14]:
class InspectableRecord(DynamicRecordV3):
    @property
    def fields(self):
        return tuple(object.__getattribute__(self, '_record').keys())

    def get_field(self, name):
        record = object.__getattribute__(self, '_record')
        try:
            return record[name]
        except KeyError:
            raise KeyError(f'No field named {name!r}') from None

In [15]:
strange = InspectableRecord({'name': 'example', 'fields': 'database value'})
print(strange.fields)
print(strange.get_field('fields'))
assert strange.get_field('fields') == 'database value'

('name', 'fields')
database value


# Problem 5 — Read-only fields

Some fields should be readable but not assignable.

Examples include IDs, creation timestamps, or database-controlled values.

We will keep a set of read-only field names as internal metadata.

In [16]:
class ReadOnlyRecord(InspectableRecord):
    def __init__(self, record, *, read_only=()):
        super().__init__(record)
        object.__setattr__(self, '_read_only', frozenset(read_only))

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)
            return

        read_only = object.__getattribute__(self, '_read_only')
        if name in read_only:
            raise AttributeError(f'Field {name!r} is read-only')

        super().__setattr__(name, value)

In [17]:
record = ReadOnlyRecord({'id': 100, 'status': 'new'}, read_only={'id'})
record.status = 'done'

try:
    record.id = 101
except AttributeError as exc:
    print('Rejected:', exc)

assert record.id == 100
assert record.status == 'done'

Rejected: Field 'id' is read-only


# Problem 6 — Define deletion semantics

Python gives us `__delattr__` for expressions such as `del record.nickname`.

For this exercise:

- internal names may be deleted normally,
- read-only fields may not be deleted,
- known writable fields are removed,
- unknown fields raise `AttributeError`.

In [18]:
class DeletableRecord(ReadOnlyRecord):
    def __delattr__(self, name):
        if name.startswith('_'):
            object.__delattr__(self, name)
            return

        read_only = object.__getattribute__(self, '_read_only')
        if name in read_only:
            raise AttributeError(f'Field {name!r} is read-only')

        record = object.__getattribute__(self, '_record')
        if name not in record:
            raise AttributeError(name)

        del record[name]

In [19]:
record = DeletableRecord({'id': 1, 'nickname': 'temporary', 'active': True}, read_only={'id'})
del record.nickname

print(record.fields)
assert 'nickname' not in record.fields
assert not hasattr(record, 'nickname')

('id', 'active')


# Problem 7 — Record an audit trail

Now we want to remember each meaningful field change.

An audit entry will contain:

- field name,
- old value,
- new value,
- timestamp.

In [20]:
@dataclass(frozen=True)
class AuditEntry:
    field: str
    old_value: Any
    new_value: Any
    changed_at: datetime

If the old and new values are equal, we will not create an audit entry.

That keeps the history focused on real changes.

In [21]:
class AuditedRecord(ReadOnlyRecord):
    def __init__(self, record, *, read_only=()):
        super().__init__(record, read_only=read_only)
        object.__setattr__(self, '_audit_log', [])

    @property
    def audit_log(self):
        return tuple(object.__getattribute__(self, '_audit_log'))

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)
            return

        record = object.__getattribute__(self, '_record')
        if name not in record:
            raise AttributeError(name)

        old_value = deepcopy(record[name])
        super().__setattr__(name, value)

        if old_value != value:
            object.__getattribute__(self, '_audit_log').append(
                AuditEntry(
                    field=name,
                    old_value=old_value,
                    new_value=deepcopy(value),
                    changed_at=datetime.now(timezone.utc),
                )
            )

In [22]:
record = AuditedRecord({'id': 1, 'status': 'new', 'count': 0}, read_only={'id'})
record.status = 'processing'
record.count = 1
record.count = 1
record.status = 'done'

for entry in record.audit_log:
    print(entry)

assert len(record.audit_log) == 3

AuditEntry(field='status', old_value='new', new_value='processing', changed_at=datetime.datetime(2026, 11, 21, 16, 57, 8, 705071, tzinfo=datetime.timezone.utc))
AuditEntry(field='count', old_value=0, new_value=1, changed_at=datetime.datetime(2026, 11, 21, 16, 57, 8, 705290, tzinfo=datetime.timezone.utc))
AuditEntry(field='status', old_value='processing', new_value='done', changed_at=datetime.datetime(2026, 11, 21, 16, 57, 8, 705500, tzinfo=datetime.timezone.utc))


Notice that the audit class delegates the real assignment to the parent class.

It does not reimplement the read-only policy.

That keeps each layer focused on one responsibility.

# Problem 8 — Snapshots

An audit trail describes changes over time.

A snapshot is different: it is a complete copy of the current state at one moment.

We will use `deepcopy` so nested mutable values are isolated.

In [23]:
class SnapshotRecord(AuditedRecord):
    def snapshot(self):
        return deepcopy(object.__getattribute__(self, '_record'))

In [24]:
record = SnapshotRecord({'name': 'sample', 'tags': ['math', 'python']})
snap = record.snapshot()
snap['tags'].append('changed outside')

print(record.tags)
print(snap['tags'])
assert record.tags == ['math', 'python']

['math', 'python']
['math', 'python', 'changed outside']


# Problem 9 — Aliases

Suppose the stored schema uses `first_name` and `last_name`, but another part of the program wants `given_name` and `family_name`.

Duplicating the values would create synchronization problems.

Instead, we can translate aliases to real field names.

In [25]:
class AliasRecord(SnapshotRecord):
    def __init__(self, record, *, aliases=None, read_only=()):
        super().__init__(record, read_only=read_only)
        object.__setattr__(self, '_aliases', dict(aliases or {}))

    def _resolve_name(self, name):
        aliases = object.__getattribute__(self, '_aliases')
        return aliases.get(name, name)

    def __getattr__(self, name):
        resolved = self._resolve_name(name)
        record = object.__getattribute__(self, '_record')
        if resolved in record:
            return record[resolved]
        raise AttributeError(name)

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)
            return
        resolved = self._resolve_name(name)
        super().__setattr__(resolved, value)

In [26]:
record = AliasRecord(
    {'first_name': 'Ada', 'last_name': 'Lovelace'},
    aliases={'given_name': 'first_name', 'family_name': 'last_name'},
)

print(record.first_name)
print(record.given_name)
record.family_name = 'King'
print(record.last_name)

assert record.given_name == 'Ada'
assert record.last_name == 'King'

Ada
Ada
King


The alias is only another name for the same stored field.

No duplicate copy of the data is created.

# Problem 10 — Unusual dictionary keys

A dictionary can contain keys such as:

- `first-name`
- `postal code`
- `2fa_enabled`

Attribute syntax cannot naturally represent every possible key.

Therefore dot access should be a convenience layer, not the only API.

In [27]:
class HybridRecord(AliasRecord):
    def __getitem__(self, key):
        return object.__getattribute__(self, '_record')[key]

    def __setitem__(self, key, value):
        record = object.__getattribute__(self, '_record')
        if key not in record:
            raise KeyError(key)

        if isinstance(key, str) and key.isidentifier():
            setattr(self, key, value)
            return

        old_value = deepcopy(record[key])
        record[key] = deepcopy(value)

        if old_value != value:
            object.__getattribute__(self, '_audit_log').append(
                AuditEntry(
                    field=key,
                    old_value=old_value,
                    new_value=deepcopy(value),
                    changed_at=datetime.now(timezone.utc),
                )
            )

In [28]:
record = HybridRecord({'first_name': 'Grace', 'first-name': 'Grace Hopper', 'postal code': '12345'})

print(record.first_name)
print(record['first-name'])
print(record['postal code'])

record['postal code'] = '54321'
assert record['postal code'] == '54321'

Grace
Grace Hopper
12345


# Problem 11 — Improve `dir()`

Dynamic fields are not automatically visible to tools that use `dir(record)`.

We can combine normal Python names with dynamic field names and aliases.

In [29]:
class DiscoverableRecord(HybridRecord):
    def __dir__(self):
        names = set(super().__dir__())
        record = object.__getattribute__(self, '_record')
        aliases = object.__getattribute__(self, '_aliases')

        names.update(name for name in record if isinstance(name, str) and name.isidentifier())
        names.update(name for name in aliases if isinstance(name, str) and name.isidentifier())
        return sorted(names)

In [30]:
record = DiscoverableRecord(
    {'first_name': 'Ada', 'last_name': 'Lovelace'},
    aliases={'given_name': 'first_name'},
)

print([name for name in ('first_name', 'given_name', 'snapshot') if name in dir(record)])

assert 'first_name' in dir(record)
assert 'given_name' in dir(record)
assert 'snapshot' in dir(record)

['first_name', 'given_name', 'snapshot']


# Problem 12 — Local edits with explicit save

If a wrapper stores the original database dictionary directly, assignment changes the backing data immediately.

Many applications prefer a different model:

1. load a local copy,
2. edit it,
3. explicitly save it.

We will also keep an `_original` baseline so we can detect changes and roll them back.

In [31]:
class BoundRecord(DiscoverableRecord):
    def __init__(self, table_data, record_id, *, aliases=None, read_only=()):
        if record_id not in table_data:
            raise KeyError(record_id)

        local = deepcopy(table_data[record_id])
        super().__init__(local, aliases=aliases, read_only=read_only)

        object.__setattr__(self, '_table_data', table_data)
        object.__setattr__(self, '_record_id', record_id)
        object.__setattr__(self, '_original', deepcopy(local))

    @property
    def record_id(self):
        return object.__getattribute__(self, '_record_id')

    @property
    def changes(self):
        original = object.__getattribute__(self, '_original')
        current = object.__getattribute__(self, '_record')
        return {
            name: (original.get(name), current.get(name))
            for name in set(original) | set(current)
            if original.get(name) != current.get(name)
        }

    @property
    def is_dirty(self):
        return bool(self.changes)

    def save(self):
        table_data = object.__getattribute__(self, '_table_data')
        record_id = object.__getattribute__(self, '_record_id')
        current = object.__getattribute__(self, '_record')

        table_data[record_id] = deepcopy(current)
        object.__setattr__(self, '_original', deepcopy(current))

    def rollback(self):
        original = object.__getattribute__(self, '_original')
        object.__setattr__(self, '_record', deepcopy(original))

In [32]:
db_copy = deepcopy(DB)
person = BoundRecord(db_copy['Person'], 1)

person.last_name = 'Local Edit'

print('local:', person.last_name)
print('database:', db_copy['Person'][1]['last_name'])
print('changes:', person.changes)

assert db_copy['Person'][1]['last_name'] == 'Newton'
assert person.is_dirty

person.save()
assert db_copy['Person'][1]['last_name'] == 'Local Edit'
assert not person.is_dirty

local: Local Edit
database: Newton
changes: {'last_name': ('Newton', 'Local Edit')}


Rollback uses the last saved baseline.

A field changed back to its original value automatically disappears from `changes` because the property compares complete states rather than maintaining a separate dirty set.

In [33]:
db_copy = deepcopy(DB)
person = BoundRecord(db_copy['Person'], 2)

person.last_name = 'Temporary'
person.active = False
print(person.changes)

person.rollback()
print(person.last_name, person.active)

assert person.last_name == 'von Leibniz'
assert person.active is True
assert not person.is_dirty

{'last_name': ('von Leibniz', 'Temporary'), 'active': (True, False)}
von Leibniz True


# Problem 13 — A callable table object

We want this interface:

```python
people = Table(DB, 'Person')
person = people(1)
```

The table wrapper validates the table name once and returns a bound record for a requested ID.

In [34]:
class Table:
    def __init__(self, db, table_name):
        if table_name not in db:
            raise ValueError(f'Unknown table: {table_name}')
        self._db = db
        self._table_name = table_name

    @property
    def table_name(self):
        return self._table_name

    def __call__(self, record_id):
        return BoundRecord(self._db[self._table_name], record_id)

    def __len__(self):
        return len(self._db[self._table_name])

In [35]:
people = Table(DB, 'Person')
person = people(3)

print(person.first_name, person.last_name)
print('rows:', len(people))

assert person.last_name == 'Fourier'
assert len(people) == 4

Joseph Fourier
rows: 4


# Problem 14 — Lazy relationships

`Person` stores `country_id`.

It would be convenient to write:

```python
person.country.name
person.country.capital
```

We will describe relationships as metadata instead of hard-coding them inside the record class.

In [36]:
class Database:
    def __init__(self, data):
        self._data = data
        self._relationships = {}

    def add_relationship(self, source_table, relation_name, fk_field, target_table):
        self._relationships[(source_table, relation_name)] = (fk_field, target_table)

A relational record first tries stored-field lookup.

Only if that fails does it check whether the requested name is a registered relationship.

In [37]:
class RelationalRecord(BoundRecord):
    def __init__(self, database, table_name, record_id):
        super().__init__(database._data[table_name], record_id)
        object.__setattr__(self, '_database', database)
        object.__setattr__(self, '_table_name', table_name)

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            pass

        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        relation = database._relationships.get((table_name, name))

        if relation is None:
            raise AttributeError(name)

        fk_field, target_table = relation
        record = object.__getattribute__(self, '_record')
        target_id = record[fk_field]

        return RelationalRecord(database, target_table, target_id)

In [38]:
rel_db = Database(deepcopy(DB))
rel_db.add_relationship('Person', 'country', 'country_id', 'Country')

person = RelationalRecord(rel_db, 'Person', 1)

print(person.first_name)
print(person.country.name)
print(person.country.capital)

assert person.country.name == 'United Kingdom'

Isaac
United Kingdom
London


When `person.country` is evaluated:

1. normal Python lookup fails,
2. dynamic stored-field lookup fails,
3. relationship metadata is checked,
4. `country_id` is read,
5. the related `Country` record is wrapped.

The relationship is loaded only when requested.

# Problem 15 — Cache relationship objects

The previous implementation creates a new wrapper on every access.

A cache can reuse the same wrapper, but then we must invalidate it when the foreign key changes.

In [39]:
class CachedRelationalRecord(RelationalRecord):
    def __init__(self, database, table_name, record_id):
        super().__init__(database, table_name, record_id)
        object.__setattr__(self, '_relation_cache', {})

    def __getattr__(self, name):
        record = object.__getattribute__(self, '_record')
        if name in record:
            return record[name]

        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        relation = database._relationships.get((table_name, name))

        if relation is None:
            raise AttributeError(name)

        cache = object.__getattribute__(self, '_relation_cache')
        if name not in cache:
            fk_field, target_table = relation
            target_id = record[fk_field]
            cache[name] = CachedRelationalRecord(database, target_table, target_id)

        return cache[name]

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)
            return

        super().__setattr__(name, value)

        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        cache = object.__getattribute__(self, '_relation_cache')

        for (source_table, relation_name), (fk_field, target_table) in database._relationships.items():
            if source_table == table_name and fk_field == name:
                cache.pop(relation_name, None)

In [40]:
rel_db = Database(deepcopy(DB))
rel_db.add_relationship('Person', 'country', 'country_id', 'Country')

person = CachedRelationalRecord(rel_db, 'Person', 1)
first = person.country
second = person.country

assert first is second

person.country_id = 5
third = person.country

print(first.name, '->', third.name)
assert first.name == 'United Kingdom'
assert third.name == 'Germany'
assert first is not third

United Kingdom -> Germany


This illustrates a general software rule:

> every cache needs an invalidation strategy.

Caching is not only about remembering a value. It is also about knowing when that value is no longer valid.

# Problem 16 — Computed fields

Some useful values do not need to be stored.

For example, a person's `full_name` can be calculated from two stored fields.

We will register computed fields as functions.

In [41]:
class ComputedDatabase(Database):
    def __init__(self, data):
        super().__init__(data)
        self._computed = {}

    def add_computed(self, table_name, field_name, function):
        self._computed[(table_name, field_name)] = function

In [42]:
class ComputedRecord(CachedRelationalRecord):
    def __getattr__(self, name):
        record = object.__getattribute__(self, '_record')
        if name in record:
            return record[name]

        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')

        function = database._computed.get((table_name, name))
        if function is not None:
            return function(self)

        return super().__getattr__(name)

In [43]:
comp_db = ComputedDatabase(deepcopy(DB))
comp_db.add_relationship('Person', 'country', 'country_id', 'Country')
comp_db.add_computed('Person', 'full_name', lambda p: f'{p.first_name} {p.last_name}')

person = ComputedRecord(comp_db, 'Person', 1)

print(person.full_name)
print(person.country.capital)

assert person.full_name == 'Isaac Newton'
assert person.country.capital == 'London'

Isaac Newton
London


The lookup order is now:

1. ordinary Python attributes,
2. stored fields,
3. computed fields,
4. relationships,
5. `AttributeError`.

That precedence should be documented because names can collide.

# Problem 17 — Query using dynamic fields

Let's add a tiny in-memory query wrapper.

The interesting part is that the field name is known only at runtime, so we use `getattr(record, field)`.

In [44]:
class QueryTable:
    def __init__(self, database, table_name):
        if table_name not in database._data:
            raise KeyError(table_name)
        self._database = database
        self._table_name = table_name

    def get(self, record_id):
        return ComputedRecord(self._database, self._table_name, record_id)

    def all(self):
        table = self._database._data[self._table_name]
        return [self.get(record_id) for record_id in sorted(table)]

    def filter_by(self, **criteria):
        matches = []
        for record in self.all():
            if all(getattr(record, field) == expected for field, expected in criteria.items()):
                matches.append(record)
        return matches

    def where(self, predicate):
        return [record for record in self.all() if predicate(record)]

In [45]:
query_db = ComputedDatabase(deepcopy(DB))
people = QueryTable(query_db, 'Person')

active_people = people.filter_by(active=True)
print([(p.first_name, p.last_name) for p in active_people])
assert len(active_people) == 3

born_before_1700 = people.where(lambda p: p.born < 1700)
print([(p.first_name, p.born) for p in born_before_1700])
assert [p.first_name for p in born_before_1700] == ['Isaac', 'Gottfried']

[('Isaac', 'Newton'), ('Gottfried', 'von Leibniz'), ('Joseph', 'Fourier')]
[('Isaac', 1642), ('Gottfried', 1646)]


# Problem 18 — Safe serialization

Returning `record.__dict__` would leak internal implementation details such as caches and backing-table references.

Serialization should return only the public data.

In [46]:
class SerializableRecord(ComputedRecord):
    def to_dict(self):
        return deepcopy(object.__getattribute__(self, '_record'))

In [47]:
serial_db = ComputedDatabase(deepcopy(DB))
person = SerializableRecord(serial_db, 'Person', 1)
data = person.to_dict()

print(data)
assert '_database' not in data
assert '_record' not in data
assert data['first_name'] == 'Isaac'

data['first_name'] = 'Changed outside'
assert person.first_name == 'Isaac'

{'first_name': 'Isaac', 'last_name': 'Newton', 'born': 1642, 'country_id': 1, 'active': True}


# Problem 19 — Why not override `__getattribute__` instead?

`__getattribute__` runs for **every** attribute lookup.

That makes it more powerful, but also easier to use incorrectly.

In [48]:
class GetAttributeDemo:
    def __init__(self):
        object.__setattr__(self, '_value', 10)

    def __getattribute__(self, name):
        print(f'lookup: {name}')
        return object.__getattribute__(self, name)

    @property
    def doubled(self):
        return self._value * 2


demo = GetAttributeDemo()
print(demo.doubled)

lookup: doubled
lookup: _value
20


Even access to `_value` passed through `__getattribute__`.

For dynamic database fields, we usually need fallback behavior, not universal interception. That is why `__getattr__` is usually the safer choice here.

# Problem 20 — Diagnose a broken implementation

Consider this class:

```python
class ProblemRecord:
    def __init__(self, record):
        self._record = record

    def __getattr__(self, name):
        return self._record[name]

    def __setattr__(self, name, value):
        self._record[name] = value
```

Try to identify the problems before reading the solution below.

## Solution

### 1. Initialization recursion

`self._record = record` invokes the custom `__setattr__` before `_record` exists.

### 2. Wrong exception type

`self._record[name]` raises `KeyError`. Missing attributes should raise `AttributeError`.

### 3. No internal namespace

Every assignment becomes a data field, so the object cannot cleanly maintain metadata.

### 4. Arbitrary field creation

The setter silently creates new keys. A stricter dynamic record may prefer to reject unknown public fields.

# Problem 21 — Nested mutable values

Consider this operation:

```python
record.tags.append('python')
```

The attribute itself was not reassigned. The returned list was mutated in place.

Therefore `__setattr__` does not see the change.

In [49]:
record = DynamicRecordV3({'tags': ['math']})
record.tags.append('python')

print(record.tags)
assert record.tags == ['math', 'python']

['math', 'python']


An audit system that watches only `__setattr__` will miss in-place mutations of nested mutable values.

Possible strategies include:

- returning defensive copies,
- using immutable nested values,
- wrapping lists/dictionaries in observable containers,
- requiring explicit mutation methods.

There is no universal answer; it depends on the abstraction you want.

# Problem 22 — Defensive reads

One strict approach is to return a deep copy whenever a dynamic field is read.

Then callers cannot mutate nested stored values accidentally.

In [50]:
class DefensiveReadRecord:
    def __init__(self, record):
        object.__setattr__(self, '_record', deepcopy(record))

    def __getattr__(self, name):
        record = object.__getattribute__(self, '_record')
        if name in record:
            return deepcopy(record[name])
        raise AttributeError(name)

In [51]:
record = DefensiveReadRecord({'tags': ['math']})
outside = record.tags
outside.append('python')

print('outside:', outside)
print('record:', record.tags)

assert outside == ['math', 'python']
assert record.tags == ['math']

outside: ['math', 'python']
record: ['math']


Defensive copying improves isolation, but it also costs more memory and CPU and makes in-place editing less convenient.

This is a design trade-off rather than a magic-method rule.

# Capstone — A compact tutorial data layer

We now combine the major ideas into a coherent design.

The final version supports:

- dynamic stored fields,
- aliases,
- read-only fields,
- audit history,
- dirty tracking,
- save/rollback,
- lazy cached relationships,
- computed fields,
- querying,
- serialization,
- and discoverable names.

Every feature below has already been introduced separately.

In [52]:
class TutorialDatabase:
    def __init__(self, data):
        self._data = data
        self._relationships = {}
        self._computed = {}
        self._aliases = {}
        self._read_only = {}

    def relationship(self, source_table, relation_name, fk_field, target_table):
        self._relationships[(source_table, relation_name)] = (fk_field, target_table)

    def computed(self, table_name, field_name, function):
        self._computed[(table_name, field_name)] = function

    def aliases(self, table_name, mapping):
        self._aliases[table_name] = dict(mapping)

    def read_only(self, table_name, fields):
        self._read_only[table_name] = frozenset(fields)

    def table(self, table_name):
        return TutorialTable(self, table_name)

In [53]:
class TutorialTable:
    def __init__(self, database, table_name):
        if table_name not in database._data:
            raise KeyError(table_name)
        self._database = database
        self._table_name = table_name

    def get(self, record_id):
        return TutorialRecord(self._database, self._table_name, record_id)

    def __call__(self, record_id):
        return self.get(record_id)

    def all(self):
        table = self._database._data[self._table_name]
        return [self.get(record_id) for record_id in sorted(table)]

    def filter_by(self, **criteria):
        return [
            record
            for record in self.all()
            if all(getattr(record, field) == value for field, value in criteria.items())
        ]

    def where(self, predicate):
        return [record for record in self.all() if predicate(record)]

In [54]:
class TutorialRecord:
    def __init__(self, database, table_name, record_id):
        table_data = database._data[table_name]
        if record_id not in table_data:
            raise KeyError(f'Record {record_id} not found in {table_name}')

        local = deepcopy(table_data[record_id])
        object.__setattr__(self, '_database', database)
        object.__setattr__(self, '_table_name', table_name)
        object.__setattr__(self, '_record_id', record_id)
        object.__setattr__(self, '_record', local)
        object.__setattr__(self, '_original', deepcopy(local))
        object.__setattr__(self, '_audit_log', [])
        object.__setattr__(self, '_relation_cache', {})

    def _resolve_name(self, name):
        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        aliases = database._aliases.get(table_name, {})
        return aliases.get(name, name)

    def __getattr__(self, name):
        resolved = self._resolve_name(name)
        record = object.__getattribute__(self, '_record')

        if resolved in record:
            return record[resolved]

        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')

        function = database._computed.get((table_name, name))
        if function is not None:
            return function(self)

        relation = database._relationships.get((table_name, name))
        if relation is not None:
            cache = object.__getattribute__(self, '_relation_cache')
            if name not in cache:
                fk_field, target_table = relation
                target_id = record[fk_field]
                if target_id is None:
                    return None
                cache[name] = TutorialRecord(database, target_table, target_id)
            return cache[name]

        raise AttributeError(f'{type(self).__name__} has no attribute {name!r}')

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)
            return

        resolved = self._resolve_name(name)
        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        record = object.__getattribute__(self, '_record')

        read_only = database._read_only.get(table_name, frozenset())
        if resolved in read_only:
            raise AttributeError(f'Field {resolved!r} is read-only')
        if resolved not in record:
            raise AttributeError(f'Unknown stored field {resolved!r}')

        old_value = deepcopy(record[resolved])
        if old_value == value:
            return

        record[resolved] = deepcopy(value)
        object.__getattribute__(self, '_audit_log').append(
            AuditEntry(
                field=resolved,
                old_value=old_value,
                new_value=deepcopy(value),
                changed_at=datetime.now(timezone.utc),
            )
        )

        cache = object.__getattribute__(self, '_relation_cache')
        for (source_table, relation_name), (fk_field, target_table) in database._relationships.items():
            if source_table == table_name and fk_field == resolved:
                cache.pop(relation_name, None)

    def __getitem__(self, key):
        return object.__getattribute__(self, '_record')[key]

    def __dir__(self):
        names = set(super().__dir__())
        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        record = object.__getattribute__(self, '_record')

        names.update(name for name in record if isinstance(name, str) and name.isidentifier())
        names.update(database._aliases.get(table_name, {}))
        names.update(field for (table, field) in database._computed if table == table_name)
        names.update(relation for (table, relation) in database._relationships if table == table_name)
        return sorted(names)

    @property
    def record_id(self):
        return object.__getattribute__(self, '_record_id')

    @property
    def audit_log(self):
        return tuple(object.__getattribute__(self, '_audit_log'))

    @property
    def changes(self):
        original = object.__getattribute__(self, '_original')
        current = object.__getattribute__(self, '_record')
        return {
            name: (original.get(name), current.get(name))
            for name in set(original) | set(current)
            if original.get(name) != current.get(name)
        }

    @property
    def is_dirty(self):
        return bool(self.changes)

    def save(self):
        database = object.__getattribute__(self, '_database')
        table_name = object.__getattribute__(self, '_table_name')
        record_id = object.__getattribute__(self, '_record_id')
        current = object.__getattribute__(self, '_record')

        database._data[table_name][record_id] = deepcopy(current)
        object.__setattr__(self, '_original', deepcopy(current))

    def rollback(self):
        original = object.__getattribute__(self, '_original')
        object.__setattr__(self, '_record', deepcopy(original))
        object.__getattribute__(self, '_relation_cache').clear()

    def snapshot(self):
        return deepcopy(object.__getattribute__(self, '_record'))

    def to_dict(self):
        return deepcopy(object.__getattribute__(self, '_record'))

    def __repr__(self):
        table_name = object.__getattribute__(self, '_table_name')
        record_id = object.__getattribute__(self, '_record_id')
        marker = ' dirty' if self.is_dirty else ''
        return f'<{type(self).__name__} {table_name}({record_id}){marker}>'

## Configure the final layer

The record class itself does not need to hard-code that a person has aliases, a country relationship, or a computed full name.

Those behaviors are registered as metadata.

In [55]:
tutorial_data = deepcopy(DB)
tutorial_db = TutorialDatabase(tutorial_data)

tutorial_db.aliases('Person', {'given_name': 'first_name', 'family_name': 'last_name'})
tutorial_db.read_only('Person', {'born'})
tutorial_db.relationship('Person', 'country', 'country_id', 'Country')
tutorial_db.computed('Person', 'full_name', lambda p: f'{p.first_name} {p.last_name}')
tutorial_db.computed('Country', 'summary', lambda c: f'{c.name} — capital: {c.capital}')

people = tutorial_db.table('Person')
countries = tutorial_db.table('Country')

## Stored fields, aliases, computed fields, and relationships

In [56]:
person = people(1)

print(person)
print(person.first_name)
print(person.given_name)
print(person.full_name)
print(person.country.summary)

assert person.first_name == 'Isaac'
assert person.given_name == 'Isaac'
assert person.full_name == 'Isaac Newton'
assert person.country.capital == 'London'

<TutorialRecord Person(1)>
Isaac
Isaac
Isaac Newton
United Kingdom — capital: London


## Read-only behavior

In [57]:
person = people(1)

try:
    person.born = 1700
except AttributeError as exc:
    print('Rejected:', exc)

assert person.born == 1642

Rejected: Field 'born' is read-only


## Dirty tracking and rollback

In [58]:
person = people(1)
person.family_name = 'Temporary'

print(person)
print(person.changes)

assert person.is_dirty
assert person.last_name == 'Temporary'

person.rollback()
assert not person.is_dirty
assert person.last_name == 'Newton'

<TutorialRecord Person(1) dirty>
{'last_name': ('Newton', 'Temporary')}


## Relationship cache invalidation

In [59]:
person = people(1)
first_country = person.country
print(first_country.name)

person.country_id = 5
second_country = person.country
print(second_country.name)

assert first_country.name == 'United Kingdom'
assert second_country.name == 'Germany'
assert first_country is not second_country

United Kingdom
Germany


## Explicit persistence

In [60]:
person = people(2)
person.last_name = 'Leibniz'

print('local:', person.last_name)
print('database:', tutorial_data['Person'][2]['last_name'])

assert tutorial_data['Person'][2]['last_name'] == 'von Leibniz'

person.save()
assert tutorial_data['Person'][2]['last_name'] == 'Leibniz'
assert not person.is_dirty

local: Leibniz
database: von Leibniz


## Query examples

In [61]:
active_people = people.filter_by(active=True)
print([p.full_name for p in active_people])
assert len(active_people) == 3

born_before_1700 = people.where(lambda p: p.born < 1700)
print([(p.full_name, p.born) for p in born_before_1700])
assert len(born_before_1700) == 2

['Isaac Newton', 'Gottfried Leibniz', 'Joseph Fourier']
[('Isaac Newton', 1642), ('Gottfried Leibniz', 1646)]


# Best-practice summary

The final class is less important than the rules used to build it.

### Prefer `__getattr__` for fallback lookup

It preserves normal Python methods, properties, and instance attributes.

### Treat `__setattr__` with care

It sees every assignment, including initialization.

### Use base implementations for trusted internal state

`object.__setattr__` and `object.__getattribute__` help avoid accidental recursion.

### Raise `AttributeError` for missing attributes

That preserves expected introspection behavior.

### Provide an explicit escape hatch

Attribute syntax cannot safely represent every possible field name. Item access or `get_field()` solves that.

### Document precedence

Stored fields, methods, computed fields, and relationships can collide.

### Every cache needs invalidation

A relationship cache becomes wrong when its foreign key changes.

### Make persistence semantics explicit

Decide whether assignment mutates the backing store immediately or only after `save()`.

### Remember nested mutable values

`__setattr__` cannot observe `record.tags.append(...)`. If that matters, you need another strategy.

# Final verification exercise

Run the next cell. If it prints `ALL TUTORIAL TESTS PASSED`, the major behaviors introduced above are working together.

In [62]:
def run_tutorial_tests():
    data = deepcopy(DB)
    database = TutorialDatabase(data)

    database.aliases('Person', {'given_name': 'first_name'})
    database.read_only('Person', {'born'})
    database.relationship('Person', 'country', 'country_id', 'Country')
    database.computed('Person', 'full_name', lambda p: f'{p.first_name} {p.last_name}')

    people = database.table('Person')
    person = people(1)

    assert person.first_name == 'Isaac'
    assert person.given_name == 'Isaac'
    assert person.full_name == 'Isaac Newton'
    assert person.country.name == 'United Kingdom'

    person.last_name = 'Edited'
    assert person.is_dirty
    assert person.changes['last_name'] == ('Newton', 'Edited')
    assert person.audit_log[-1].field == 'last_name'

    person.rollback()
    assert person.last_name == 'Newton'
    assert not person.is_dirty

    old_country = person.country
    person.country_id = 5
    new_country = person.country
    assert old_country.name == 'United Kingdom'
    assert new_country.name == 'Germany'
    assert old_country is not new_country

    try:
        person.born = 1700
    except AttributeError:
        pass
    else:
        raise AssertionError('read-only field was modified')

    person.last_name = 'Newton-Smith'
    assert data['Person'][1]['last_name'] == 'Newton'
    person.save()
    assert data['Person'][1]['last_name'] == 'Newton-Smith'
    assert not person.is_dirty

    active = people.filter_by(active=True)
    assert len(active) == 3

    print('ALL TUTORIAL TESTS PASSED')


run_tutorial_tests()

ALL TUTORIAL TESTS PASSED
